# Transformation Pipeline Validation & EDA

This notebook validates the data transformation pipeline and identifies areas for future transformations.

**Usage**: Simply run all cells. The notebook will automatically find and load the processed dataset from `data/processed/`.

## Objectives
1. Validate all transformation functions work correctly
2. Analyze transformation outcomes and flag distributions
3. Identify data quality patterns requiring new transformations
4. Document findings for future pipeline improvements

In [ ]:
import sys
import os
import glob
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from data.transformations import (
    transform_dataframe,
    clean_first_name,
    clean_last_name,
    clean_middle_name,
    clean_suffix,
    clean_birth_date,
    clean_ssn,
    clean_email,
    clean_phone,
    clean_address,
    clean_city,
    clean_zip,
    clean_state,
    clean_sex_at_birth,
    process_name_redistribution,
    detect_camelcase,
    validate_name_presence
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

PROCESSED_DATA_DIR = '../data/processed'

---
## 0. Auto-Load Processed Dataset

In [ ]:
def find_and_load_dataset(data_dir):
    """
    Automatically find and load the processed dataset.
    Supports CSV, Parquet, and Pickle formats.
    If multiple files exist, loads the most recently modified one.
    """
    supported_extensions = ['*.csv', '*.parquet', '*.pkl', '*.pickle']
    all_files = []
    
    for ext in supported_extensions:
        all_files.extend(glob.glob(os.path.join(data_dir, ext)))
        all_files.extend(glob.glob(os.path.join(data_dir, '**', ext), recursive=True))
    
    # Filter out __init__.py and other non-data files
    data_files = [f for f in all_files if not f.endswith('__init__.py') and os.path.getsize(f) > 0]
    
    if not data_files:
        print(f"[!] No data files found in {data_dir}")
        print(f"    Supported formats: CSV, Parquet, Pickle")
        print(f"    Please place your processed dataset in the data/processed/ folder.")
        return None, None
    
    # Sort by modification time (most recent first)
    data_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    
    selected_file = data_files[0]
    file_ext = os.path.splitext(selected_file)[1].lower()
    file_size = os.path.getsize(selected_file) / (1024 * 1024)  # MB
    mod_time = datetime.fromtimestamp(os.path.getmtime(selected_file))
    
    print(f"Found {len(data_files)} data file(s). Loading most recent:")
    print(f"  File: {os.path.basename(selected_file)}")
    print(f"  Size: {file_size:.2f} MB")
    print(f"  Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    # Load based on file type
    try:
        if file_ext == '.csv':
            df = pd.read_csv(selected_file, low_memory=False)
        elif file_ext == '.parquet':
            df = pd.read_parquet(selected_file)
        elif file_ext in ['.pkl', '.pickle']:
            df = pd.read_pickle(selected_file)
        else:
            print(f"[!] Unsupported file format: {file_ext}")
            return None, None
        
        print(f"Successfully loaded {len(df):,} records with {len(df.columns)} columns")
        return df, selected_file
        
    except Exception as e:
        print(f"[!] Error loading file: {e}")
        return None, None

# Load the processed dataset
df, data_file_path = find_and_load_dataset(PROCESSED_DATA_DIR)

if df is not None:
    print(f"\nDataset Preview:")
    display(df.head(3))
    print(f"\nColumn names:")
    print(list(df.columns))

In [ ]:
# Check if data was loaded, if not stop execution
if df is None:
    raise SystemExit("No data file found. Please add a processed dataset to data/processed/ and re-run.")

print(f"Proceeding with validation on {len(df):,} records...")

---
## 1. Unit Tests for Individual Transformations

Test each transformation function with known inputs to validate correct behavior.

### 1.1 SSN Transformation Tests

In [ ]:
ssn_test_cases = [
    # (input, expected_output, expected_flags, description)
    ('123-45-6789', None, {'is_junk_SSN': True}, 'Sequential pattern should be junk'),
    ('123456789', None, {'is_junk_SSN': True}, 'Sequential without dashes'),
    ('987654321', None, {'is_junk_SSN': True}, 'Descending sequential'),
    ('000-12-3456', None, {'is_invalid_SSN': True}, 'Invalid area number 000'),
    ('666-12-3456', None, {'is_invalid_SSN': True}, 'Invalid area number 666'),
    ('900-12-3456', None, {'is_invalid_SSN': True}, 'ITIN range 9XX'),
    ('123-00-4567', None, {'is_invalid_SSN': True}, 'Invalid group number 00'),
    ('123-45-0000', None, {'is_invalid_SSN': True}, 'Invalid serial number 0000'),
    ('111111111', None, {'is_junk_SSN': True}, 'Repeating digits'),
    ('010101010', None, {'is_junk_SSN': True}, 'Known junk alternating'),
    ('111223333', None, {'is_junk_SSN': True}, 'Woolworth wallet card'),
    ('456-78-9012', '456789012', {}, 'Valid SSN'),
    ('', None, {'is_missing_SSN': True}, 'Empty string'),
    (None, None, {'is_missing_SSN': True}, 'None value'),
    ('1234567', '001234567', {'PADDED_SSN': True}, '7-digit padded'),
]

print("SSN Transformation Test Results:")
print("=" * 80)
all_passed = True
for inp, expected_out, expected_flags, desc in ssn_test_cases:
    result, flags = clean_ssn(inp)
    
    # Check output
    output_match = (pd.isna(result) and expected_out is None) or (result == expected_out)
    
    # Check flags
    flags_match = all(flags.get(k, False) == v for k, v in expected_flags.items())
    
    passed = output_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected_out}, Got: {result}")
        print(f"       Expected flags: {expected_flags}, Got: {flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.2 Email Transformation Tests

In [ ]:
email_test_cases = [
    # (input, expected_output, expected_flags, description)
    ('test@example.com', None, {'is_junk_Email': True}, 'Junk domain example.com'),
    ('noemail@gmail.com', None, {'is_junk_Email': True}, 'Junk prefix noemail'),
    ('unknown@unknown.com', None, {'is_junk_Email': True}, 'Known junk exact match'),
    ('ab@gmail.com', None, {'is_junk_Email': True}, 'Local part too short (2 chars)'),
    ('user123456@gmail.com', None, {'is_junk_Email': True}, 'Sequential digits in local'),
    ('valid.user@company.org', 'valid.user@company.org', {}, 'Valid email'),
    ('VALID@DOMAIN.COM', 'valid@domain.com', {}, 'Uppercase normalized'),
    ('missingat.com', None, {'is_invalid_Email': True}, 'Missing @ symbol'),
    ('nodomain@', None, {'is_invalid_Email': True}, 'Missing domain'),
    ('', None, {'is_missing_Email': True}, 'Empty string'),
    (None, None, {'is_missing_Email': True}, 'None value'),
    ('nan', None, {'is_junk_Email': True}, 'Primitive placeholder nan'),
]

print("Email Transformation Test Results:")
print("=" * 80)
all_passed = True
for inp, expected_out, expected_flags, desc in email_test_cases:
    result, flags = clean_email(inp)
    
    output_match = (pd.isna(result) and expected_out is None) or (result == expected_out)
    flags_match = all(flags.get(k, False) == v for k, v in expected_flags.items())
    
    passed = output_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected_out}, Got: {result}")
        print(f"       Expected flags: {expected_flags}, Got relevant: {flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.3 Name Redistribution Tests

In [ ]:
name_test_cases = [
    # (first, middle, last, suffix, expected_first, expected_middle, expected_last, description)
    ('JOHN MICHAEL SMITH', None, None, None, 'JOHN', 'MICHAEL', 'SMITH', '3-word first, empty last'),
    ('JOHN SMITH', None, None, None, 'JOHN', None, 'SMITH', '2-word first, empty last'),
    ('WILLIAM EARL', None, 'JOHNSON', None, 'WILLIAM', 'EARL', 'JOHNSON', '2-word first, last exists, move to middle'),
    ('MARY ANN', 'LOUISE', 'SMITH', None, 'MARY ANN', 'LOUISE', 'SMITH', '2-word first but middle exists, keep as-is'),
    ('JOHN DE LA CRUZ', None, None, None, 'JOHN', None, 'DE LA CRUZ', 'Compound last name prefix'),
    ('JOHN SMITH JR', None, None, None, 'JOHN', None, 'SMITH', 'Extract suffix during redistribution'),
]

print("Name Redistribution Test Results:")
print("=" * 80)
all_passed = True
for first, middle, last, suffix, exp_first, exp_middle, exp_last, desc in name_test_cases:
    res_first, res_middle, res_last, res_suffix, indicators = process_name_redistribution(first, middle, last, suffix)
    
    first_match = (pd.isna(res_first) and exp_first is None) or (res_first == exp_first)
    middle_match = (pd.isna(res_middle) and exp_middle is None) or (res_middle == exp_middle)
    last_match = (pd.isna(res_last) and exp_last is None) or (res_last == exp_last)
    
    passed = first_match and middle_match and last_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: first={repr(first)}, middle={repr(middle)}, last={repr(last)}")
        print(f"       Expected: first={exp_first}, middle={exp_middle}, last={exp_last}")
        print(f"       Got: first={res_first}, middle={res_middle}, last={res_last}")
        print(f"       Indicators: {indicators}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.4 CamelCase Detection Tests

In [ ]:
camelcase_test_cases = [
    ('JohnSmith', ['JOHN', 'SMITH'], 'Simple CamelCase'),
    ('WilliamEarl', ['WILLIAM', 'EARL'], 'Two names'),
    ('MariaDelCarmen', ['MARIA', 'DEL', 'CARMEN'], 'Three parts'),
    ('JOHN', [], 'All uppercase - no CamelCase'),
    ('john', [], 'All lowercase - no CamelCase'),
    ('John', [], 'Single word with capital'),
    ('JohnMichaelSmith', ['JOHN', 'MICHAEL', 'SMITH'], 'Three names'),
]

print("CamelCase Detection Test Results:")
print("=" * 80)
all_passed = True
for inp, expected, desc in camelcase_test_cases:
    result = detect_camelcase(inp)
    passed = result == expected
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: {repr(inp)}")
        print(f"       Expected: {expected}, Got: {result}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

### 1.5 Name Presence Validation Tests

In [ ]:
name_presence_test_cases = [
    ('JOHN', 'SMITH', True, {}, 'Both names present'),
    (None, 'SMITH', False, {'MISSING_FIRST_NAME': True}, 'Missing first name'),
    ('JOHN', None, False, {'MISSING_LAST_NAME': True}, 'Missing last name'),
    (None, None, False, {'MISSING_BOTH_NAMES': True}, 'Both names missing'),
    ('', 'SMITH', False, {'MISSING_FIRST_NAME': True}, 'Empty first name'),
    ('JOHN', '', False, {'MISSING_LAST_NAME': True}, 'Empty last name'),
]

print("Name Presence Validation Test Results:")
print("=" * 80)
all_passed = True
for first, last, exp_valid, exp_flags, desc in name_presence_test_cases:
    is_valid, flags = validate_name_presence(first, last)
    
    valid_match = is_valid == exp_valid
    flags_match = all(flags.get(k, False) == v for k, v in exp_flags.items())
    
    passed = valid_match and flags_match
    status = "PASS" if passed else "FAIL"
    all_passed = all_passed and passed
    
    print(f"[{status}] {desc}")
    if not passed:
        print(f"       Input: first={repr(first)}, last={repr(last)}")
        print(f"       Expected valid={exp_valid}, flags={exp_flags}")
        print(f"       Got valid={is_valid}, flags={flags}")

print("=" * 80)
print(f"Overall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")

---
## 2. Run Transformation Pipeline on Loaded Data

In [ ]:
# Check if data is already transformed (has _clean columns)
clean_cols = [c for c in df.columns if '_clean' in c.lower()]
is_already_transformed = len(clean_cols) > 0

if is_already_transformed:
    print(f"Dataset appears to be already transformed ({len(clean_cols)} _clean columns found)")
    print(f"Using existing transformed data for validation.")
    transformed_df = df.copy()
else:
    print(f"Running transformation pipeline on {len(df):,} records...")
    print("This may take a few minutes for large datasets.")
    print()
    
    import time
    start_time = time.time()
    transformed_df = transform_dataframe(df)
    elapsed = time.time() - start_time
    
    print(f"Transformation complete in {elapsed:.1f} seconds.")

print(f"\nOutput has {len(transformed_df.columns)} columns.")

# Show new columns added
new_cols = [c for c in transformed_df.columns if c not in df.columns]
if new_cols:
    print(f"\nNew columns added ({len(new_cols)}):")
    for col in sorted(new_cols)[:20]:
        print(f"  - {col}")
    if len(new_cols) > 20:
        print(f"  ... and {len(new_cols) - 20} more")

---
## 3. Transformation Outcomes Analysis

### 3.1 ValidRecord Analysis

In [ ]:
print("ValidRecord Distribution:")
print(transformed_df['ValidRecord'].value_counts())
print(f"\nInvalid records: {(~transformed_df['ValidRecord']).sum()} / {len(transformed_df)}")

invalid_records = transformed_df[~transformed_df['ValidRecord']]
if len(invalid_records) > 0:
    print("\nInvalid records details:")
    display_cols = ['PATID', 'FirstNM', 'LastNM', 'FirstNM_clean', 'LastNM_clean', 
                    'MISSING_FIRST_NAME', 'MISSING_LAST_NAME']
    display_cols = [c for c in display_cols if c in invalid_records.columns]
    print(invalid_records[display_cols].to_string())

### 3.2 Name Redistribution Analysis

In [ ]:
name_indicators = ['INFERRED_LAST_FROM_FIRST', 'MOVED_MIDDLE_FROM_FIRST', 
                   'CAMELCASE_SPLIT', 'EXTRACTED_SUFFIX', 'needs_name_review']
name_indicators = [c for c in name_indicators if c in transformed_df.columns]

print("Name Redistribution Flag Counts:")
for col in name_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

# Show records that were redistributed
redistributed = transformed_df[transformed_df[name_indicators].any(axis=1)]
if len(redistributed) > 0:
    print(f"\n{len(redistributed)} records had name redistribution:")
    display_cols = ['PATID', 'FirstNM', 'MiddleNM', 'LastNM', 
                    'FirstNM_clean', 'MiddleNM_clean', 'LastNM_clean'] + name_indicators
    display_cols = [c for c in display_cols if c in redistributed.columns]
    print(redistributed[display_cols].to_string())

### 3.3 SSN Validation Analysis

In [ ]:
ssn_indicators = ['is_missing_SSN', 'is_junk_SSN', 'is_invalid_SSN', 'PADDED_SSN']
ssn_indicators = [c for c in ssn_indicators if c in transformed_df.columns]

print("SSN Validation Flag Counts:")
for col in ssn_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

print("\nSSN Cleaning Results:")
ssn_cols = ['PATID', 'SSN', 'SSN_clean', 'SSN_Last4'] + ssn_indicators
ssn_cols = [c for c in ssn_cols if c in transformed_df.columns]
print(transformed_df[ssn_cols].to_string())

### 3.4 Email Validation Analysis

In [ ]:
email_indicators = ['is_missing_Email', 'is_junk_Email', 'is_invalid_Email']
email_indicators = [c for c in email_indicators if c in transformed_df.columns]

print("Email Validation Flag Counts:")
for col in email_indicators:
    count = transformed_df[col].sum()
    print(f"  {col}: {count}")

print("\nEmail Cleaning Results:")
email_cols = ['PATID', 'Email', 'Email_clean'] + email_indicators
email_cols = [c for c in email_cols if c in transformed_df.columns]
print(transformed_df[email_cols].to_string())

### 3.5 Quality Score Distribution

In [ ]:
if 'QUALITY_SCORE' in transformed_df.columns:
    print("Quality Score Statistics:")
    print(transformed_df['QUALITY_SCORE'].describe())
    
    print("\nQuality Tier Distribution:")
    quality_flags = ['LOW_QUALITY_RECORD', 'PARTIAL_IDENTITY_RECORD', 'HIGH_CONFIDENCE_DEMOGRAPHICS']
    for flag in quality_flags:
        if flag in transformed_df.columns:
            print(f"  {flag}: {transformed_df[flag].sum()}")

---
## 4. Data Overview & Column Analysis

In [ ]:
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total Records: {len(transformed_df):,}")
print(f"Total Columns: {len(transformed_df.columns)}")
print(f"Memory Usage: {transformed_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()

# Categorize columns
original_cols = [c for c in transformed_df.columns if not any(x in c for x in ['_clean', 'is_', 'INVALID', 'JUNK', 'MISSING', 'QUALITY', 'HAS_', 'PCT_'])]
clean_cols = [c for c in transformed_df.columns if '_clean' in c]
flag_cols = [c for c in transformed_df.columns if any(x in c for x in ['is_', 'INVALID', 'JUNK', 'MISSING', 'HAS_', 'needs_'])]

print(f"Original data columns: {len(original_cols)}")
print(f"Cleaned value columns: {len(clean_cols)}")
print(f"Flag/indicator columns: {len(flag_cols)}")

In [ ]:
# Show null rates for key columns
print("\nNULL RATES FOR KEY COLUMNS")
print("=" * 60)

key_clean_cols = ['FirstNM_clean', 'LastNM_clean', 'BirthDT_clean', 'SSN_clean', 
                  'Email_clean', 'AddressLine1_clean', 'CityNM_clean', 'StateCD_clean', 'ZipCD_base']
key_clean_cols = [c for c in key_clean_cols if c in transformed_df.columns]

for col in key_clean_cols:
    null_count = transformed_df[col].isna().sum()
    null_pct = null_count / len(transformed_df) * 100
    print(f"  {col}: {null_count:,} nulls ({null_pct:.1f}%)")

---
## 5. Future Transformation Discovery

Analyze patterns in the data that may require new transformation rules.

### 5.1 Identify Frequent Values Needing Review

In [ ]:
def analyze_frequent_values(df, column, top_n=20, threshold_pct=0.5):
    """Analyze most frequent values in a column for potential junk patterns."""
    if column not in df.columns:
        print(f"Column {column} not found")
        return None
    
    # Get non-null values only
    non_null = df[column].dropna()
    if len(non_null) == 0:
        print(f"Column {column} has no non-null values")
        return None
    
    value_counts = non_null.value_counts().head(top_n)
    print(f"\nTop {top_n} values in {column} ({len(non_null):,} non-null values):")
    
    # Display with percentages
    for val, count in value_counts.items():
        pct = count / len(non_null) * 100
        print(f"  {repr(val)[:50]}: {count:,} ({pct:.2f}%)")
    
    # Flag potential issues
    threshold = len(df) * (threshold_pct / 100)
    suspicious = [(val, count) for val, count in value_counts.items() if count > threshold]
    
    if suspicious:
        print(f"\n[!] Values appearing in >{threshold_pct}% of records (review for potential junk):")
        for val, count in suspicious:
            print(f"    '{val}': {count:,} records ({count/len(df)*100:.2f}%)")
    
    return value_counts

# Analyze key cleaned columns
print("FREQUENT VALUE ANALYSIS")
print("=" * 60)

analyze_frequent_values(transformed_df, 'Email_clean', threshold_pct=0.1)

In [ ]:
# Analyze other key columns
for col in ['FirstNM_clean', 'LastNM_clean', 'CityNM_clean']:
    if col in transformed_df.columns:
        analyze_frequent_values(transformed_df, col, top_n=15, threshold_pct=1.0)

### 5.2 Pattern Analysis for New Junk Detection

In [ ]:
def find_suspicious_patterns(df, column, min_count=5):
    """
    Find suspicious patterns in values that passed cleaning.
    Returns values that appear frequently and may need new rules.
    """
    if column not in df.columns:
        return {}
    
    clean_col = f"{column}_clean" if f"{column}_clean" in df.columns else column
    
    # Get non-null values
    values = df[clean_col].dropna()
    
    # Find repeated values
    value_counts = values.value_counts()
    repeated = value_counts[value_counts >= min_count]
    
    return repeated

print("Checking for suspicious patterns that passed validation...")

# Check emails
suspicious_emails = find_suspicious_patterns(transformed_df, 'Email', min_count=2)
if len(suspicious_emails) > 0:
    print(f"\nEmails appearing multiple times (potential shared/fake):")
    print(suspicious_emails.to_string())

### 5.3 Identify Missing Transformation Categories

In [ ]:
def analyze_nullified_values(original_df, transformed_df, column):
    """
    Analyze what values were nullified during transformation.
    Helps identify patterns that might need explicit handling.
    """
    clean_col = f"{column}_clean" if f"{column}_clean" in transformed_df.columns else column
    
    if column not in original_df.columns or clean_col not in transformed_df.columns:
        return None
    
    # Find records where original had value but cleaned is null
    had_value = original_df[column].notna()
    now_null = transformed_df[clean_col].isna()
    nullified = had_value & now_null
    
    if nullified.sum() > 0:
        nullified_values = original_df.loc[nullified, column].value_counts()
        print(f"\n{column}: {nullified.sum()} values nullified during cleaning:")
        print(nullified_values.head(20).to_string())
    else:
        print(f"\n{column}: No values were nullified")
    
    return nullified

# Analyze nullified values for key fields
for col in ['SSN', 'Email', 'FirstNM', 'AddressLine1']:
    analyze_nullified_values(sample_data, transformed_df, col)

---
## 6. Summary & Recommendations

Document findings and suggest future improvements.

In [ ]:
print("=" * 80)
print("TRANSFORMATION PIPELINE VALIDATION REPORT")
print("=" * 80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data Source: {os.path.basename(data_file_path) if data_file_path else 'Unknown'}")
print()

# Collect stats
stats = {}
stats['Total Records'] = f"{len(transformed_df):,}"

if 'ValidRecord' in transformed_df.columns:
    valid_count = transformed_df['ValidRecord'].sum()
    invalid_count = (~transformed_df['ValidRecord']).sum()
    stats['Valid Records'] = f"{valid_count:,} ({valid_count/len(transformed_df)*100:.1f}%)"
    stats['Invalid Records'] = f"{invalid_count:,} ({invalid_count/len(transformed_df)*100:.1f}%)"

print("RECORD VALIDITY")
print("-" * 40)
for key, value in stats.items():
    print(f"  {key}: {value}")

# SSN stats
print("\nSSN VALIDATION")
print("-" * 40)
if 'SSN_clean' in transformed_df.columns:
    valid_ssn = transformed_df['SSN_clean'].notna().sum()
    print(f"  Valid SSNs: {valid_ssn:,} ({valid_ssn/len(transformed_df)*100:.1f}%)")
    if 'is_missing_SSN' in transformed_df.columns:
        missing = transformed_df['is_missing_SSN'].sum()
        print(f"  Missing SSNs: {missing:,} ({missing/len(transformed_df)*100:.1f}%)")
    if 'is_junk_SSN' in transformed_df.columns:
        junk = transformed_df['is_junk_SSN'].sum()
        print(f"  Junk SSNs: {junk:,} ({junk/len(transformed_df)*100:.1f}%)")
    if 'is_invalid_SSN' in transformed_df.columns:
        invalid = transformed_df['is_invalid_SSN'].sum()
        print(f"  Invalid SSNs: {invalid:,} ({invalid/len(transformed_df)*100:.1f}%)")

# Email stats
print("\nEMAIL VALIDATION")
print("-" * 40)
if 'Email_clean' in transformed_df.columns:
    valid_email = transformed_df['Email_clean'].notna().sum()
    print(f"  Valid Emails: {valid_email:,} ({valid_email/len(transformed_df)*100:.1f}%)")
    if 'is_missing_Email' in transformed_df.columns:
        missing = transformed_df['is_missing_Email'].sum()
        print(f"  Missing Emails: {missing:,} ({missing/len(transformed_df)*100:.1f}%)")
    if 'is_junk_Email' in transformed_df.columns:
        junk = transformed_df['is_junk_Email'].sum()
        print(f"  Junk Emails: {junk:,} ({junk/len(transformed_df)*100:.1f}%)")

# Name stats
print("\nNAME PROCESSING")
print("-" * 40)
if 'FirstNM_clean' in transformed_df.columns:
    valid_first = transformed_df['FirstNM_clean'].notna().sum()
    valid_last = transformed_df['LastNM_clean'].notna().sum() if 'LastNM_clean' in transformed_df.columns else 0
    print(f"  Valid First Names: {valid_first:,} ({valid_first/len(transformed_df)*100:.1f}%)")
    print(f"  Valid Last Names: {valid_last:,} ({valid_last/len(transformed_df)*100:.1f}%)")

if 'INFERRED_LAST_FROM_FIRST' in transformed_df.columns:
    inferred = transformed_df['INFERRED_LAST_FROM_FIRST'].sum()
    print(f"  Names Redistributed: {inferred:,}")
if 'CAMELCASE_SPLIT' in transformed_df.columns:
    camel = transformed_df['CAMELCASE_SPLIT'].sum()
    print(f"  CamelCase Splits: {camel:,}")
if 'needs_name_review' in transformed_df.columns:
    review = transformed_df['needs_name_review'].sum()
    print(f"  Needs Manual Review: {review:,}")

# Quality score
print("\nQUALITY SCORES")
print("-" * 40)
if 'QUALITY_SCORE' in transformed_df.columns:
    print(f"  Mean Quality Score: {transformed_df['QUALITY_SCORE'].mean():.2f}")
    print(f"  Median Quality Score: {transformed_df['QUALITY_SCORE'].median():.2f}")
    if 'HIGH_CONFIDENCE_DEMOGRAPHICS' in transformed_df.columns:
        high_conf = transformed_df['HIGH_CONFIDENCE_DEMOGRAPHICS'].sum()
        print(f"  High Confidence Records: {high_conf:,} ({high_conf/len(transformed_df)*100:.1f}%)")
    if 'LOW_QUALITY_RECORD' in transformed_df.columns:
        low_qual = transformed_df['LOW_QUALITY_RECORD'].sum()
        print(f"  Low Quality Records: {low_qual:,} ({low_qual/len(transformed_df)*100:.1f}%)")

In [ ]:
print("\n" + "=" * 80)
print("RECOMMENDATIONS FOR FUTURE TRANSFORMATIONS")
print("=" * 80)

recommendations = []

# Check for records needing review
if 'needs_name_review' in transformed_df.columns:
    review_count = transformed_df['needs_name_review'].sum()
    if review_count > 0:
        recommendations.append(f"1. REVIEW {review_count:,} records flagged with 'needs_name_review'\n"
                              f"   - CamelCase splits may have edge cases\n"
                              f"   - Long concatenated names may need manual review")

# Check for frequent emails
if 'Email_clean' in transformed_df.columns:
    email_counts = transformed_df['Email_clean'].dropna().value_counts()
    frequent_emails = email_counts[email_counts >= 50]
    if len(frequent_emails) > 0:
        recommendations.append(f"2. REVIEW {len(frequent_emails)} emails appearing 50+ times\n"
                              f"   - May indicate clinic-default placeholders not yet in junk list\n"
                              f"   - Top: {list(frequent_emails.head(3).index)}")

# Check invalid record rate
if 'ValidRecord' in transformed_df.columns:
    invalid_pct = (~transformed_df['ValidRecord']).sum() / len(transformed_df) * 100
    if invalid_pct > 5:
        recommendations.append(f"3. HIGH INVALID RATE: {invalid_pct:.1f}% of records are invalid\n"
                              f"   - Review invalid record patterns\n"
                              f"   - Consider if validation rules are too strict")

# Standard recommendations
recommendations.append("4. CONSIDER adding nickname/diminutive mapping\n"
                      "   - BILL -> WILLIAM, BOB -> ROBERT, etc.\n"
                      "   - Would improve blocking scheme recall")

recommendations.append("5. VALIDATE compound name prefix list completeness\n"
                      "   - Current list: DE LA, VAN, MC, etc.\n"
                      "   - May need expansion based on population demographics")

for rec in recommendations:
    print(f"\n{rec}")

In [ ]:
print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)
print(f"\nTransformed DataFrame shape: {transformed_df.shape[0]:,} rows x {transformed_df.shape[1]} columns")
print(f"Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")